# Itinerary Generator Pipeline — Step-by-Step

Walks through `services/trip-generation/` one phase at a time. Run cells
top-to-bottom the first time; afterwards you can tweak any intermediate
variable and re-run just the downstream phase.

**Phases**
1. Route search & composition
2. POI retrieval + enrichment + union with route POIs
   - 2a. Vector search → `{ pois, queryEmbedding }`
   - 2b. Union with route POIs
   - 2c. Enrich with live data (Google Places)
   - **2d. Re-rank via MLP scorer** ← new (no-op until `models/poi-reranker-weights.json` exists)
3. LLM pick (Gemini)
4. Solver — order stops, enforce opening hours & meal anchors
5. Persist (optional — writes to Supabase)
6. **Record training feedback** ← new (writes to `itinerary_feedback` for re-ranker training)

**Before you start:** see `notebooks/README.md` for tslab install. Launch
`jupyter lab` from the **repo root**, not from `notebooks/`.

## Setup — load `.env.local` and sanity-check keys

In [1]:
import { dirname, resolve } from "node:path";
import { existsSync } from "node:fs";

let repoRoot = process.cwd();
while (!existsSync(resolve(repoRoot, ".env.local")) && dirname(repoRoot) !== repoRoot) {
  repoRoot = dirname(repoRoot);
}

const envPath = resolve(repoRoot, ".env.local");
if (!existsSync(envPath)) {
  throw new Error(`Missing .env.local above ${process.cwd()}. Did you start jupyter inside this repo?`);
}

if (process.cwd() !== repoRoot) {
  process.chdir(repoRoot);
}
import "tsx/cjs";
process.loadEnvFile(envPath);

const required = [
  "NEXT_PUBLIC_SUPABASE_URL",
  "SUPABASE_SECRET_KEY",
  "GEMINI_API_KEY",
];
const missing = required.filter((k) => !process.env[k]);
if (missing.length) throw new Error(`Missing env vars: ${missing.join(", ")}`);

console.log("repo root:", repoRoot);
console.log("env loaded:", required.map((k) => `${k}=${process.env[k]?.slice(0, 8)}...`).join("  "));
console.log("GOOGLE_PLACES_API_KEY:", process.env.GOOGLE_PLACES_API_KEY ? "set" : "(missing - enrichment will be skipped)");


repo root: E:\Projects\travel-sync-ai
env loaded: NEXT_PUBLIC_SUPABASE_URL=https://...  SUPABASE_SECRET_KEY=sb_secre...  GEMINI_API_KEY=AIzaSyCx...
GOOGLE_PLACES_API_KEY: set


## Imports — pipeline functions

Path alias `@/*` is wired through `notebooks/tsconfig.json` so we can use
the same imports the production code does.

In [ ]:
const routeEngine = require("@/services/trip-generation/route-engine");
const poiEngine = require("@/services/trip-generation/poi-engine");
const rerankerModule = require("@/services/trip-generation/reranker");
const solver = require("@/services/trip-generation/solver");
const orchestrator = require("@/services/trip-generation/orchestrator");
const { randomBytes } = require("node:crypto");

Object.assign(globalThis, {
  searchRoutesByVibe: routeEngine.searchRoutesByVibe,
  composeFromRoutes: routeEngine.composeFromRoutes,
  searchPoisByVibe: poiEngine.searchPoisByVibe,
  buildVibeQuery: poiEngine.buildVibeQuery,
  loadPoisByIds: poiEngine.loadPoisByIds,
  enrichWithLiveData: poiEngine.enrichWithLiveData,
  rerankCandidates: rerankerModule.rerankCandidates,
  recordFeedback: rerankerModule.recordFeedback,
  solveItinerary: solver.solveItinerary,
  PACE_CAPS: solver.PACE_CAPS,
  __notebook: orchestrator.__notebook,
});

const genId = randomBytes(4).toString("hex");
(globalThis as any).genId = genId;
console.log("genId for this notebook run:", genId);

## Pre-flight — load curated Niseko data and seed routes

Phase 1a needs `route_templates` rows for the destination (with matching
pace + alias) and Phase 2b needs the route's `place_ids` to resolve in
`poi_embeddings` or the health filter drops them.

This cell idempotently seeds the example trip using the **curated dataset
under `data/japan-ski-trip/niseko/`** (resorts, restaurants, activities).
It builds two multi-stop balanced-pace routes — Hirafu day and Niseko
Village day — that each compose a complete ski-day-out: morning ski →
lunch → afternoon onsen → dinner. Restaurant opening hours are pinned to
the solver's meal-anchor windows (lunch opens 12:00, dinner opens 18:00)
so the curator-ordered routes are guaranteed feasible.

If the routes already exist for this destination, the cell short-circuits
into a no-op. If you've already run `scripts/ingest-ski-dataset.ts` +
`scripts/seed-route-templates.ts`, those rows live under
`Hokkaido, Japan` and don't collide with this `Niseko, Japan` seed.

In [3]:
import { readFileSync } from "node:fs";
import { resolve as _resolve } from "node:path";
import { createAdminClient } from "@/lib/db";
import { generateEmbedding as _genEmb } from "@/lib/gemini";
import { normalizeAliases } from "@/lib/destination-aliases";

const _SEED_DEST = "Niseko, Japan";
const _SEED_ALIASES = normalizeAliases([
  "Niseko, Japan", "Niseko", "ニセコ",
  "Hokkaido, Japan", "Hokkaido", "北海道",
  "Japan", "Japan ski", "ski japan",
]);

// Curated dataset shipped in the repo (no network calls).
const _dataDir = _resolve(process.cwd(), "data/japan-ski-trip/niseko");
const _resortsRaw = JSON.parse(readFileSync(_resolve(_dataDir, "resorts.json"), "utf8"));
const _restaurantsRaw = JSON.parse(readFileSync(_resolve(_dataDir, "restaurants.json"), "utf8"));
const _activitiesRaw = JSON.parse(readFileSync(_resolve(_dataDir, "activities.json"), "utf8"));

interface _Resort { id: string; name: string; name_ja: string; lat: number; lng: number; night_skiing: boolean; notes: string }
interface _Restaurant { id: string; name: string; cuisine: string; lat: number; lng: number; notes: string }
interface _Activity { id: string; name: string; type: string; lat: number; lng: number; notes: string }

const _resorts = _resortsRaw.resorts as _Resort[];
const _restaurants = _restaurantsRaw.restaurants as _Restaurant[];
const _activities = _activitiesRaw.activities as _Activity[];

// 7-day-a-week opening periods.
const _allDays = (open: number, close: number) =>
  Array.from({ length: 7 }, (_, d) => ({ openDay: d, openMinutes: open, closeDay: d, closeMinutes: close }));

const _resortPeriods = (nightSki: boolean) => {
  const days = _allDays(8*60+30, 16*60+30);
  if (nightSki) days.push(..._allDays(17*60, 20*60+30));
  return days;
};
// Restaurant hours pinned to the solver's meal anchors. Real Niseko venues
// typically open 11:30 / 17:30, but the solver enforces arrive ∈ [12:00,14:00]
// for lunch and [18:00,20:00] for dinner — opening exactly on those edges
// lets the curator order [activity, restaurant] without manual padding stops.
const _LUNCH_PERIODS = _allDays(12*60, 15*60);
const _DINNER_PERIODS = _allDays(18*60, 22*60);
const _ONSEN_PERIODS = _allDays(10*60, 22*60);

// Multi-stop balanced-pace routes; preorderedDays=true keeps the solver from
// re-permuting them, so the listed order *is* the schedule.
const _ROUTES: Array<{ title: string; summary: string; place_ids: string[] }> = [
  {
    title: "Hirafu 滑雪一日",
    summary:
      "Niseko Grand Hirafu 早場滑雪、Tsubara Tsubara 湯咖哩午餐、街中 Yukoro 溫泉、Bang Bang 串燒晚餐。" +
      "Best-known Niseko United base + classic apres-ski walking circuit, all within Hirafu Village.",
    place_ids: ["niseko-grand-hirafu", "tsubara-tsubara", "yukoro-onsen", "bang-bang"],
  },
  {
    title: "Niseko Village 寬鬆滑雪日",
    summary:
      "Niseko Village 滑雪、Rakuichi 手打蕎麥午餐、昆布溫泉 Tsuruga 露天浴、Ezo Seafoods 北海道海鮮晚餐。" +
      "Quieter side of Niseko United for relaxed couples; ends in an onsen and oysters.",
    place_ids: ["niseko-village", "rakuichi-soba", "niseko-konbu-onsen-tsuruga", "ezo-seafoods"],
  },
];

const _LUNCH_IDS = new Set(["tsubara-tsubara", "rakuichi-soba", "graubunden"]);
const _DINNER_IDS = new Set(["bang-bang", "ezo-seafoods", "abucha-2", "the-barn-by-odin"]);

const _db = createAdminClient();

const _routeUpsertConflictHint = (message: string) =>
  message.includes("no unique or exclusion constraint")
    ? `${message}. Apply migration supabase/migrations/20260526000000_route_templates_plain_upsert_conflict.sql, or run: create unique index if not exists route_templates_destination_title_plain_uniq on route_templates (destination_name, title);`
    : message;

// No-op when both routes already present.
const { data: _existingRoutes } = await _db
  .from("route_templates")
  .select("title")
  .eq("destination_name", _SEED_DEST)
  .eq("is_archived", false);
const _haveTitles = new Set((_existingRoutes ?? []).map((r) => r.title));
const _missingRoutes = _ROUTES.filter((r) => !_haveTitles.has(r.title));

if (_missingRoutes.length === 0) {
  console.log(`pre-flight: routes already present for ${_SEED_DEST}, skipping`);
} else {
  // Only seed the POIs referenced by the missing routes.
  const _neededPoiIds = new Set<string>();
  for (const r of _missingRoutes) for (const id of r.place_ids) _neededPoiIds.add(id);

  type _Row = {
    place_id: string; destination_name: string; destination_aliases: string[];
    name: string; item_type: string; tags: string[]; description: string;
    lat: number; lng: number; live_data: object; _embedText: string;
  };
  const _poiRows: _Row[] = [];

  for (const r of _resorts) {
    if (!_neededPoiIds.has(r.id)) continue;
    _poiRows.push({
      place_id: r.id,
      destination_name: _SEED_DEST,
      destination_aliases: normalizeAliases([..._SEED_ALIASES, r.id, r.name, r.name_ja]),
      name: r.name,
      item_type: "activity",
      tags: ["ski", "snow", "winter", "adventure", ...(r.night_skiing ? ["nightlife"] : [])],
      description: r.notes,
      lat: r.lat,
      lng: r.lng,
      live_data: {
        placeId: r.id, name: r.name, address: null, rating: null, priceLevel: null,
        lat: r.lat, lng: r.lng, openingPeriods: _resortPeriods(r.night_skiing),
      },
      _embedText: `Ski resort in ${_SEED_DEST}. ${r.name} (${r.name_ja}). ${r.notes}`,
    });
  }
  for (const rt of _restaurants) {
    if (!_neededPoiIds.has(rt.id)) continue;
    const isLunch = _LUNCH_IDS.has(rt.id);
    const isDinner = _DINNER_IDS.has(rt.id);
    if (!isLunch && !isDinner) {
      throw new Error(`restaurant ${rt.id} is not classified as lunch or dinner; classify in _LUNCH_IDS/_DINNER_IDS`);
    }
    _poiRows.push({
      place_id: rt.id,
      destination_name: _SEED_DEST,
      destination_aliases: normalizeAliases([..._SEED_ALIASES, rt.id, rt.name]),
      name: rt.name,
      item_type: "restaurant",
      tags: ["food", rt.cuisine, isLunch ? "lunch" : "dinner"],
      description: rt.notes,
      lat: rt.lat,
      lng: rt.lng,
      live_data: {
        placeId: rt.id, name: rt.name, address: null, rating: null, priceLevel: null,
        lat: rt.lat, lng: rt.lng,
        openingPeriods: isLunch ? _LUNCH_PERIODS : _DINNER_PERIODS,
      },
      _embedText: `Restaurant in ${_SEED_DEST}. ${rt.name} (${rt.cuisine}). ${rt.notes}`,
    });
  }
  for (const a of _activities) {
    if (!_neededPoiIds.has(a.id)) continue;
    _poiRows.push({
      place_id: a.id,
      destination_name: _SEED_DEST,
      destination_aliases: normalizeAliases([..._SEED_ALIASES, a.id, a.name]),
      name: a.name,
      item_type: "activity",
      tags: ["onsen", "relaxed", "winter"],
      description: a.notes,
      lat: a.lat,
      lng: a.lng,
      live_data: {
        placeId: a.id, name: a.name, address: null, rating: null, priceLevel: null,
        lat: a.lat, lng: a.lng, openingPeriods: _ONSEN_PERIODS,
      },
      _embedText: `Onsen experience in ${_SEED_DEST}. ${a.name}. ${a.notes}`,
    });
  }

  const _haveIds = new Set(_poiRows.map((r) => r.place_id));
  const _missingIds = Array.from(_neededPoiIds).filter((id) => !_haveIds.has(id));
  if (_missingIds.length) {
    throw new Error(`pre-flight: no curated record for ${_missingIds.join(", ")}`);
  }

  console.log(`pre-flight: seeding ${_poiRows.length} POIs + ${_missingRoutes.length} routes for ${_SEED_DEST}`);

  for (const p of _poiRows) {
    const embedding = await _genEmb(p._embedText);
    const { _embedText: _omit, ...rest } = p;
    const { error } = await _db.from("poi_embeddings").upsert(
      { ...rest, embedding, source: "notebook-seed", last_seen_at: new Date().toISOString() },
      { onConflict: "place_id" }
    );
    if (error) throw new Error(`poi upsert failed for ${p.place_id}: ${error.message}`);
    console.log(`  + poi ${p.place_id} (${p.item_type})`);
  }

  for (const r of _missingRoutes) {
    const text =
      `Travel experiences in ${_SEED_DEST} for a balanced-paced, mid-budget trip ` +
      `with a ski, snow, onsen, winter vibe. ${r.title}. ${r.summary}`;
    const embedding = await _genEmb(text);
    const { error } = await _db.from("route_templates").upsert(
      {
        destination_name: _SEED_DEST,
        destination_aliases: _SEED_ALIASES,
        title: r.title,
        summary: r.summary,
        vibe_tags: ["ski", "snow", "winter", "onsen", "relaxed", "food"],
        pace: "balanced",
        place_ids: r.place_ids,
        pinned_vibes: ["ski"],
        embedding,
        source: "notebook-seed",
        last_health_ok_at: new Date().toISOString(),
      },
      { onConflict: "destination_name,title", ignoreDuplicates: false }
    );
    if (error) throw new Error(`route upsert failed for ${r.title}: ${_routeUpsertConflictHint(error.message)}`);
    console.log(`  + route "${r.title}" (${r.place_ids.length} stops)`);
  }
  console.log("pre-flight: done");
}


pre-flight: routes already present for Niseko, Japan, skipping


## Phase 0 — Survey input

Edit this cell to change the destination, party, vibe, etc., then re-run
everything below. The defaults below match the pre-flight seed (Niseko,
2-day couple ski/snow/onsen, balanced pace) so Phase 1a clears the 0.72
gate and Phase 4 produces a feasible 4-stop day for each of the two
days. If you change the destination, make sure the corresponding
`route_templates` + `poi_embeddings` rows exist — otherwise Phase 1a
returns 0 and Phase 2a falls back to Google Places.

In [4]:
const answers: SurveyAnswers = {
  destination: "Niseko, Japan",
  duration_days: 2,
  party: "couple",
  party_size: 2,
  budget_tier: "mid",
  vibe: ["ski", "snow", "onsen"],
  pace: "balanced",
  must_haves: null,
};

const input: GenerateInput = {
  answers,
  authorLineUserId: "U_notebook_dev",
  startDate: undefined, // defaults to today + 14d
};

__notebook.validateAnswers(input);
const startWeekday = __notebook.deriveStartWeekday(input.startDate);
Object.assign(globalThis, { answers, input, startWeekday });
console.log("validated. startWeekday =", startWeekday, " (0=Sun … 6=Sat)");


validated. startWeekday = 3  (0=Sun … 6=Sat)


## Phase 1a — Search curated routes

Vector-search `route_templates` by vibe. Returns up to 10 candidate
single-day routes, scored on similarity + boost + quality + pinned vibes.
Gate is 0.72 — anything below is filtered out.

In [5]:
const _phase1Answers = (globalThis as any).answers as SurveyAnswers | undefined;
const _phase1SearchRoutesByVibe = (globalThis as any).searchRoutesByVibe;
const _phase1GenId = (globalThis as any).genId;
if (!_phase1SearchRoutesByVibe || !_phase1GenId) {
  throw new Error("Phase 1a needs the Imports cell. Run Setup and Imports before Phase 1a after starting or restarting the kernel.");
}
if (!_phase1Answers) {
  throw new Error("Phase 1a needs `answers`. Run Phase 0 — Survey input first after starting or restarting the kernel.");
}

const routes = await _phase1SearchRoutesByVibe({
  destination: _phase1Answers.destination!,
  vibe: _phase1Answers.vibe,
  pace: _phase1Answers.pace,
  budget: _phase1Answers.budget_tier,
  k: 10,
  genId: _phase1GenId,
});
(globalThis as any).routes = routes;

console.log(`routes found: ${routes.length}`);
console.table(routes.slice(0, 10).map((r) => ({
  routeId: r.routeId.slice(0, 8),
  title: r.title,
  score: r.finalScore.toFixed(3),
  similarity: r.similarity.toFixed(3),
  places: r.placeIds.length,
})));


{"level":"info","msg":"[route-engine] scored & gated","ts":"2026-05-27T02:51:07.899Z","genId":"6df3bd97","rows":2,"passingGate":2,"gate":0.72,"topScores":"Niseko Village 寬鬆滑雪日=0.983 | Hirafu 滑雪一日=0.982"}
routes found: 2
┌─────────┬────────────┬─────────────────────────────┬─────────┬────────────┬────────┐
│ (index) │ routeId    │ title                       │ score   │ similarity │ places │
├─────────┼────────────┼─────────────────────────────┼─────────┼────────────┼────────┤
│ 0       │ 'dc7d402a' │ 'Niseko Village 寬鬆滑雪日' │ '0.983' │ '0.883'    │ 4      │
│ 1       │ '3a477aa5' │ 'Hirafu 滑雪一日'           │ '0.982' │ '0.882'    │ 4      │
└─────────┴────────────┴─────────────────────────────┴─────────┴────────────┴────────┘


## Phase 1b — Compose routes into day-slots

Greedy packer assigns routes to days, avoiding place_id collisions. Any
day that isn't covered drops into `uncoveredDays`, which Phase 3 (LLM)
will fill.

In [6]:
const _phase1bRoutes = (globalThis as any).routes;
const _phase1bAnswers = (globalThis as any).answers as SurveyAnswers | undefined;
const _phase1bComposeFromRoutes = (globalThis as any).composeFromRoutes;
if (!_phase1bComposeFromRoutes) throw new Error("Phase 1b needs the Imports cell. Run Setup and Imports first.");
if (!_phase1bAnswers) throw new Error("Phase 1b needs `answers`. Run Phase 0 first.");
if (!_phase1bRoutes) throw new Error("Phase 1b needs `routes`. Run Phase 1a first.");

const compose = _phase1bComposeFromRoutes(_phase1bRoutes, _phase1bAnswers.duration_days!);
(globalThis as any).compose = compose;

console.log("covered days:");
for (const [day, route] of compose.coveredDays) {
  console.log(`  D${day}  ${route.title}  (${route.placeIds.length} stops)`);
}
console.log("uncovered days (LLM will pick for these):", compose.uncoveredDays);
console.log("place_ids reserved by routes:", compose.usedPlaceIds.size);


covered days:
  D1  Niseko Village 寬鬆滑雪日  (4 stops)
  D2  Hirafu 滑雪一日  (4 stops)
uncovered days (LLM will pick for these): []
place_ids reserved by routes: 8


## Phase 2a — Retrieve POI candidates

Vector ANN search on `poi_embeddings`. Falls back to Google Places text
search if the corpus is cold for this destination.

In [ ]:
const _phase2aAnswers = (globalThis as any).answers as SurveyAnswers | undefined;
const _phase2aSearchPoisByVibe = (globalThis as any).searchPoisByVibe;
const _phase2aGenId = (globalThis as any).genId;
if (!_phase2aSearchPoisByVibe || !_phase2aGenId) throw new Error("Phase 2a needs the Imports cell. Run Setup and Imports first.");
if (!_phase2aAnswers) throw new Error("Phase 2a needs `answers`. Run Phase 0 first.");

// searchPoisByVibe now returns { pois, queryEmbedding } so the orchestrator
// can pass the embedding to rerankCandidates without a second Gemini call.
const { pois: poiCandidates, queryEmbedding } = await _phase2aSearchPoisByVibe({
  destination: _phase2aAnswers.destination!,
  vibe: _phase2aAnswers.vibe,
  pace: _phase2aAnswers.pace,
  budget: _phase2aAnswers.budget_tier,
  k: 30,
  genId: _phase2aGenId,
});
(globalThis as any).poiCandidates = poiCandidates;
(globalThis as any).queryEmbedding = queryEmbedding;

console.log(`POI candidates: ${poiCandidates.length} | queryEmbedding: ${queryEmbedding ? `${queryEmbedding.length}-dim` : "null (fallback path)"}`);
console.table(poiCandidates.slice(0, 15).map((p) => ({
  name: p.name,
  type: p.itemType,
  similarity: p.similarity.toFixed(3),
  tags: (p.tags ?? []).slice(0, 4).join(","),
})));

## Phase 2b — Union with route POIs

Routes lock specific place_ids; we materialize them as POI candidates
and merge into the shortlist. Route POIs take precedence.

## Phase 2d — Re-rank candidates via MLP scorer

Loads `models/poi-reranker-weights.json` (if present) and re-scores every
shortlist candidate using:

```
features = [queryEmb ‖ poiEmb ‖ hadamard(queryEmb, poiEmb)]  →  2304-dim
→ Linear(128) → GELU → Linear(1) → sigmoid
```

When the weights file is absent (cold start), this is a **no-op** —
`rankedPoiCandidates` equals `poiCandidates` and the table below shows
the original cosine-similarity order unchanged.

To train the model once you have ≥ 500 feedback rows:
```bash
SUPABASE_URL=... SUPABASE_SECRET_KEY=... GEMINI_API_KEY=... \
  python scripts/train-reranker.py --epochs 40
# → writes models/poi-reranker-weights.json
```

In [ ]:
const _phase2dPoiCandidates = (globalThis as any).poiCandidates;
const _phase2dQueryEmbedding = (globalThis as any).queryEmbedding as number[] | null;
const _phase2dRerankCandidates = (globalThis as any).rerankCandidates;
const _phase2dGenId = (globalThis as any).genId;
if (!_phase2dRerankCandidates || !_phase2dGenId) throw new Error("Phase 2d needs the Imports cell. Run Setup and Imports first.");
if (!_phase2dPoiCandidates) throw new Error("Phase 2d needs `poiCandidates`. Run Phase 2a first.");

const rankedPoiCandidates = await _phase2dRerankCandidates(
  _phase2dQueryEmbedding,
  _phase2dPoiCandidates,
  _phase2dGenId
);
(globalThis as any).rankedPoiCandidates = rankedPoiCandidates;

const weightsLoaded = _phase2dQueryEmbedding !== null &&
  rankedPoiCandidates[0]?.similarity !== _phase2dPoiCandidates[0]?.similarity;

console.log(weightsLoaded
  ? "Re-ranker active — order changed from baseline similarity."
  : "Re-ranker inactive (weights absent or queryEmbedding null) — order unchanged."
);

// Side-by-side comparison: original rank vs. re-ranked position
const originalIdx = new Map(_phase2dPoiCandidates.map((p, i) => [p.placeId, i]));
console.table(rankedPoiCandidates.slice(0, 15).map((p, newRank) => ({
  newRank,
  oldRank: originalIdx.get(p.placeId) ?? "?",
  moved: (() => {
    const old = originalIdx.get(p.placeId);
    if (old == null) return "?";
    const delta = old - newRank;
    return delta > 0 ? `▲${delta}` : delta < 0 ? `▼${Math.abs(delta)}` : "—";
  })(),
  name: p.name,
  type: p.itemType,
  score: p.similarity.toFixed(3),
})));

In [ ]:
const _phase2bCompose = (globalThis as any).compose;
// Use re-ranked candidates when Phase 2d ran; fall back to original order.
const _phase2bPoiCandidates = (globalThis as any).rankedPoiCandidates ?? (globalThis as any).poiCandidates;
const _phase2bLoadPoisByIds = (globalThis as any).loadPoisByIds;
const _phase2bNotebook = (globalThis as any).__notebook;
const _phase2bGenId = (globalThis as any).genId;
if (!_phase2bLoadPoisByIds || !_phase2bNotebook || !_phase2bGenId) throw new Error("Phase 2b needs the Imports cell. Run Setup and Imports first.");
if (!_phase2bCompose) throw new Error("Phase 2b needs `compose`. Run Phase 1b first.");
if (!_phase2bPoiCandidates) throw new Error("Phase 2b needs `poiCandidates` or `rankedPoiCandidates`. Run Phase 2a (and optionally 2d) first.");

const routePois = await _phase2bLoadPoisByIds(Array.from(_phase2bCompose.usedPlaceIds), _phase2bGenId);
const allCandidates = _phase2bNotebook.unionByPlaceId(routePois, _phase2bPoiCandidates);
(globalThis as any).routePois = routePois;
(globalThis as any).allCandidates = allCandidates;

const source = (globalThis as any).rankedPoiCandidates ? "re-ranked" : "original";
console.log(`route POIs: ${routePois.length} | search POIs: ${_phase2bPoiCandidates.length} (${source}) | unioned: ${allCandidates.length}`);
if (allCandidates.length === 0) throw new Error("no candidates — pipeline would abort with no_candidates");

## Phase 2c — Enrich with live data (Google Places)

Batch-fetches address, coords, and opening periods. The solver requires
coords + opening hours; without them, a POI is effectively unschedulable.

In [9]:
const _phase2cAllCandidates = (globalThis as any).allCandidates;
const _phase2cEnrichWithLiveData = (globalThis as any).enrichWithLiveData;
if (!_phase2cEnrichWithLiveData) throw new Error("Phase 2c needs the Imports cell. Run Setup and Imports first.");
if (!_phase2cAllCandidates) throw new Error("Phase 2c needs `allCandidates`. Run Phase 2b first.");

const enriched = await _phase2cEnrichWithLiveData(_phase2cAllCandidates);
(globalThis as any).enriched = enriched;

const withCoords = enriched.filter((e) => e.lat != null && e.lng != null).length;
const withHours = enriched.filter((e) => (e.live?.openingPeriods?.length ?? 0) > 0).length;
console.log(`enriched: ${enriched.length} total | ${withCoords} with coords | ${withHours} with opening hours`);
console.table(enriched.slice(0, 10).map((e) => ({
  name: e.name,
  type: e.itemType,
  coords: e.lat != null ? `${e.lat.toFixed(3)},${e.lng!.toFixed(3)}` : "(none)",
  hours: e.live?.openingPeriods?.length ?? 0,
  address: e.live?.address?.slice(0, 40) ?? "(none)",
})));


enriched: 34 total | 31 with coords | 8 with opening hours
┌─────────┬─────────────────────────────────────────────┬──────────────┬──────────────────┬───────┬──────────┐
│ (index) │ name                                        │ type         │ coords           │ hours │ address  │
├─────────┼─────────────────────────────────────────────┼──────────────┼──────────────────┼───────┼──────────┤
│ 0       │ 'Bang Bang'                                 │ 'restaurant' │ '42.863,140.710' │ 7     │ '(none)' │
│ 1       │ 'Ezo Seafoods'                              │ 'restaurant' │ '42.863,140.709' │ 7     │ '(none)' │
│ 2       │ 'Niseko Grand Hirafu'                       │ 'activity'   │ '42.863,140.707' │ 14    │ '(none)' │
│ 3       │ 'Tsuruga Besso Moku-no-Sho (day use onsen)' │ 'activity'   │ '42.807,140.724' │ 7     │ '(none)' │
│ 4       │ 'Niseko Village'                            │ 'activity'   │ '42.839,140.683' │ 14    │ '(none)' │
│ 5       │ 'Rakuichi Soba'                          

## Phase 3 — LLM pick (Gemini)

Gemini sees only place_id, name, type, tags, and a short summary — never
coords or hours. It returns `{ title, summary, tags, days: [{day_number,
place_ids[]}] }`. The orchestrator filters hallucinated/duplicate IDs and
truncates to the pace cap before returning.

We only ask the LLM to cover days that routes didn't already claim.

In [10]:
const _phase3Compose = (globalThis as any).compose;
const _phase3Input = (globalThis as any).input as GenerateInput | undefined;
const _phase3Answers = (globalThis as any).answers as SurveyAnswers | undefined;
const _phase3Enriched = (globalThis as any).enriched;
const _phase3GenId = (globalThis as any).genId;
const _phase3Notebook = (globalThis as any).__notebook;
if (!_phase3Notebook || !_phase3GenId) throw new Error("Phase 3 needs the Imports cell. Run Setup and Imports first.");
if (!_phase3Answers || !_phase3Input) throw new Error("Phase 3 needs `answers` and `input`. Run Phase 0 first.");
if (!_phase3Compose) throw new Error("Phase 3 needs `compose`. Run Phase 1b first.");
if (!_phase3Enriched) throw new Error("Phase 3 needs `enriched`. Run Phase 2c first.");

let pick = null;

if (_phase3Compose.uncoveredDays.length === 0) {
  console.log("all days route-covered — skipping LLM. Synthesizing title/summary from routes.");
  pick = _phase3Notebook.synthesizePickFromRoutes(_phase3Compose, _phase3Answers.destination!);
} else {
  pick = await _phase3Notebook.llmPickAssignment(
    _phase3Input,
    _phase3Enriched,
    [], // no prior infeasibility issues on first attempt
    undefined, // no prior pick
    {
      onlyDays: _phase3Compose.uncoveredDays,
      excludePlaceIds: _phase3Compose.usedPlaceIds,
      genId: _phase3GenId,
      attempt: 0,
    }
  );
}
(globalThis as any).pick = pick;

console.log("title  :", pick.title);
console.log("summary:", pick.summary);
console.log("tags   :", pick.tags.join(", "));
const byId = new Map(_phase3Enriched.map((p) => [p.placeId, p]));
(globalThis as any).byId = byId;
for (const d of pick.days) {
  const names = d.place_ids.map((id) => byId.get(id)?.name ?? `(unknown ${id.slice(0,8)})`);
  console.log(`  D${d.day_number}: ${names.join(" → ")}`);
}


all days route-covered — skipping LLM. Synthesizing title/summary from routes.
title  : Niseko, Japan — Niseko Village 寬鬆滑雪日 + more
summary: Niseko Village 滑雪、Rakuichi 手打蕎麥午餐、昆布溫泉 Tsuruga 露天浴、Ezo Seafoods 北海道海鮮晚餐。Quieter side of Niseko United for relaxed couples; ends in an onsen and oysters. Niseko Grand Hirafu 早場滑雪、Tsubara Tsubara 湯咖哩午餐、街中 Yukoro 溫泉、Bang Bang 串燒晚餐。Best-known Niseko United base + classic apres-ski walking circuit, all within Hirafu Village.
tags   : ski, snow, winter, onsen, relaxed, food


### (Optional) Hand-edit the LLM pick

If you want to override the assignment before the solver runs, mutate
`pick.days` here. Example: force a different place_id on day 2.

```ts
// pick!.days = pick!.days.map(d => d.day_number === 2
//   ? { ...d, place_ids: [enriched[0].placeId, enriched[3].placeId] }
//   : d
// );
```

## Phase 4 — Solver

Brute-force permutes each day's stops (≤6 → ≤720 permutations), simulates
travel via Haversine, enforces opening hours and meal anchors (lunch
12–14, dinner 18–20). Returns either `feasible` with timed stops or
`infeasible` with issues — in the real pipeline, infeasibility triggers a
repair loop (LLM swap + route demotion, ≤2 attempts).

In [11]:
const _phase4Pick = (globalThis as any).pick;
const _phase4Enriched = (globalThis as any).enriched;
const _phase4Input = (globalThis as any).input as GenerateInput | undefined;
const _phase4StartWeekday = (globalThis as any).startWeekday;
const _phase4Compose = (globalThis as any).compose;
const _phase4ById = (globalThis as any).byId ?? new Map(_phase4Enriched?.map((p) => [p.placeId, p]) ?? []);
const _phase4Notebook = (globalThis as any).__notebook;
if (!_phase4Notebook) throw new Error("Phase 4 needs the Imports cell. Run Setup and Imports first.");
if (!_phase4Pick) throw new Error("Phase 4 needs `pick`. Run Phase 3 first.");
if (!_phase4Enriched) throw new Error("Phase 4 needs `enriched`. Run Phase 2c first.");
if (!_phase4Input || _phase4StartWeekday == null) throw new Error("Phase 4 needs `input` and `startWeekday`. Run Phase 0 first.");
if (!_phase4Compose) throw new Error("Phase 4 needs `compose`. Run Phase 1b first.");

const solved = _phase4Notebook.trySolve(_phase4Pick, _phase4Enriched, _phase4Input, _phase4StartWeekday, _phase4Compose);
(globalThis as any).solved = solved;

if (solved.kind === "feasible") {
  console.log("FEASIBLE\n");
  for (const day of solved.days) {
    console.log(`Day ${day.dayNumber}`);
    for (const s of day.stops) {
      const arrive = `${String(Math.floor(s.arriveMinutes/60)).padStart(2,"0")}:${String(s.arriveMinutes%60).padStart(2,"0")}`;
      const depart = `${String(Math.floor(s.departMinutes/60)).padStart(2,"0")}:${String(s.departMinutes%60).padStart(2,"0")}`;
      console.log(`  ${arrive}–${depart}  ${s.poi.name}  [${s.poi.itemType}]`);
    }
    console.log("");
  }
} else {
  console.log("INFEASIBLE — issues:");
  for (const issue of solved.issues) {
    console.log(`  D${issue.dayNumber} / ${issue.reason}: ${issue.detail}`);
    if (issue.offendingPlaceIds.length) {
      const names = issue.offendingPlaceIds.map((id) => _phase4ById.get(id)?.name ?? id.slice(0,8));
      console.log(`    offending: ${names.join(", ")}`);
    }
  }
  console.log("\nTo simulate the repair loop, re-run Phase 3 with these issues passed in as the 3rd arg of llmPickAssignment, and `pick` as the 4th arg.");
}


FEASIBLE

Day 1
  09:00–10:30  Niseko Village  [activity]
  12:00–13:15  Rakuichi Soba  [restaurant]
  13:33–15:03  Tsuruga Besso Moku-no-Sho (day use onsen)  [activity]
  18:00–19:15  Ezo Seafoods  [restaurant]

Day 2
  09:00–10:30  Niseko Grand Hirafu  [activity]
  12:00–13:15  Tsubara Tsubara  [restaurant]
  13:25–14:55  Yukoro Onsen  [activity]
  18:00–19:15  Bang Bang  [restaurant]



### (Optional) Manual repair attempt

Uncomment to re-ask the LLM with the infeasibility report. The orchestrator
does this automatically up to `MAX_REPAIR_ATTEMPTS = 2` times.

```ts
// if (solved.kind === "infeasible" && pick) {
//   const repairedPick = await __notebook.llmPickAssignment(
//     input,
//     enriched,
//     solved.issues.filter((i) => !compose.coveredDays.has(i.dayNumber)),
//     pick,
//     { onlyDays: compose.uncoveredDays, excludePlaceIds: compose.usedPlaceIds, genId, attempt: 1 }
//   );
//   const resolved = __notebook.trySolve(repairedPick, enriched, input, startWeekday, compose);
//   console.log("after repair:", resolved.kind);
// }
```

## Phase 5 — Persist (commented out)

Writes a real `trip_templates` + `trip_template_versions` +
`trip_template_items` row to Supabase. **Uncomment only when you want a
real artifact.**

In [12]:
// if (solved.kind === "feasible" && pick) {
//   const out = await __notebook.persistTemplate(input, pick, solved.days, enriched, compose);
//   console.log("persisted:", out);
// } else {
//   console.log("skipping persist — solver was not feasible.");
// }


## Phase 6 — Record training feedback

Preview the rows that would be written to `itinerary_feedback` — one per
shortlist candidate, labelled with whether the LLM pick selected it.
These rows become training data for `scripts/train-reranker.py`.

The actual insert is commented out. Uncomment the `recordFeedback()` call
when you want this notebook run to count toward training data.

**Only works if Phase 3 ran the LLM** (i.e. there were uncovered days).
If all days were route-covered, `pick.days` is empty and there is nothing
to label — skip this cell.

In [ ]:
const _phase6Pick = (globalThis as any).pick;
const _phase6PoiCandidates = (globalThis as any).poiCandidates; // original order, not re-ranked
const _phase6Answers = (globalThis as any).answers as SurveyAnswers | undefined;
const _phase6BuildVibeQuery = (globalThis as any).buildVibeQuery;
const _phase6RecordFeedback = (globalThis as any).recordFeedback;
const _phase6GenId = (globalThis as any).genId;
if (!_phase6RecordFeedback || !_phase6BuildVibeQuery || !_phase6GenId) throw new Error("Phase 6 needs the Imports cell. Run Setup and Imports first.");
if (!_phase6Answers || !_phase6PoiCandidates) throw new Error("Phase 6 needs `answers` and `poiCandidates`. Run Phase 0 and Phase 2a first.");
if (!_phase6Pick) throw new Error("Phase 6 needs `pick`. Run Phase 3 first.");

const selectedIds = new Set(_phase6Pick.days.flatMap((d) => d.place_ids));
const vibeQuery = _phase6BuildVibeQuery({
  destination: _phase6Answers.destination,
  vibe: _phase6Answers.vibe,
  pace: _phase6Answers.pace,
  budget: _phase6Answers.budget_tier,
});

// Preview the feedback rows that would be written
const previewRows = _phase6PoiCandidates.map((poi, rank) => ({
  rank,
  place_id: poi.placeId.slice(0, 20),
  name: poi.name.slice(0, 30),
  similarity: poi.similarity.toFixed(3),
  was_selected: selectedIds.has(poi.placeId),
}));

console.log(`vibe_query: "${vibeQuery}"`);
console.log(`\nFeedback preview — ${previewRows.length} rows (${previewRows.filter(r => r.was_selected).length} positive, ${previewRows.filter(r => !r.was_selected).length} negative):`);
console.table(previewRows.slice(0, 20));

// ── Uncomment to actually write to Supabase ──────────────────────────────────
// const versionId = (globalThis as any).persistedVersionId; // set by Phase 5 if uncommented
// if (!versionId) {
//   console.warn("No versionId — run Phase 5 (persist) and capture out.versionId first.");
// } else {
//   await _phase6RecordFeedback({
//     genId: _phase6GenId,
//     versionId,
//     vibeQuery,
//     poiCandidates: _phase6PoiCandidates,
//     pick: _phase6Pick,
//     answers: _phase6Answers,
//   });
//   console.log("Feedback written to itinerary_feedback.");
// }
// ─────────────────────────────────────────────────────────────────────────────

console.log("\nTo train the re-ranker once you have ≥ 500 rows:");
console.log("  python scripts/train-reranker.py --epochs 40");
console.log("  # → models/poi-reranker-weights.json (drop on server, picked up on cold start)");